# Experiments

This notebook runs the retrieval pipelines and writes cached result files to `results/cache/`.

Use this notebook for:
- loading datasets and the Terrier index
- building the retrieval pipelines
- generating cached runs for DL19 and DL20
- generating the RM3 parameter sweep caches

Use `analysis.ipynb` for evaluation, per-query comparisons, and query-level analysis.

## Setup

In [59]:
from pathlib import Path

import pyterrier as pt

from src.config import (
    CACHE_PREFIX,
    FAST_MODE,
    FB_TERMS,
    FB_VALUES,
    N_TOPICS,
    NUM_RESULTS,
    RERANK_K,
)
from src.pipelines import (
    build_bm25,
    build_bm25_rm3,
    build_bm25_rm3_tasb,
    build_bm25_tasb,
    evaluate_runs,
    init_pyterrier,
    run_and_cache,
    sanitize_topics,
    subset_topics,
)

init_pyterrier()
CACHE_DIR = Path("results/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## Load dataset

In [60]:
dataset = pt.get_dataset("msmarco_passage")
irds_dataset = pt.get_dataset("irds:msmarco-passage")

# On Windows, loading topics directly from the judged DL19 subset can trigger
# file-locking issues inside ir_datasets. Load the full topics and filter them
# using the judged qrels instead.
dl19_topics_ds = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl19_qrels_ds = pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020/judged")

topics_19 = sanitize_topics(dl19_topics_ds.get_topics())
qrels_19 = dl19_qrels_ds.get_qrels()
topics_19 = topics_19[topics_19["qid"].astype(str).isin(qrels_19["qid"].astype(str))].copy()

topics_20 = sanitize_topics(dl20.get_topics())
qrels_20 = dl20.get_qrels()

print(f"DL 2019 - topics: {len(topics_19)}, qrels: {len(qrels_19)}")
print(f"DL 2020 - topics: {len(topics_20)}, qrels: {len(qrels_20)}")

DL 2019 - topics: 43, qrels: 9260
DL 2020 - topics: 54, qrels: 11386


## Load index

In [61]:
index = pt.IndexFactory.of(dataset.get_index(variant="terrier_stemmed"))
print(index.getCollectionStatistics().toString())

Number of documents: 8841823
Number of terms: 1170682
Number of postings: 215238456
Number of fields: 1
Number of tokens: 288759529
Field names: [text]
Positions:   false



## Run configuration

In [62]:
topics_19_run, qrels_19_run = subset_topics(topics_19, qrels_19, n_topics=N_TOPICS)
topics_20_run, qrels_20_run = subset_topics(topics_20, qrels_20, n_topics=N_TOPICS)

print(f"FAST_MODE={FAST_MODE}")
print(f"CACHE_PREFIX={CACHE_PREFIX}")
print(f"DL19 run topics: {len(topics_19_run)}")
print(f"DL20 run topics: {len(topics_20_run)}")

FAST_MODE=False
CACHE_PREFIX=full
DL19 run topics: 43
DL20 run topics: 54


## Build core pipelines

In [63]:
bm25 = build_bm25(index, num_results=NUM_RESULTS)
bm25_rm3 = build_bm25_rm3(index, num_results=NUM_RESULTS, fb_terms=FB_TERMS)
bm25_tasb = build_bm25_tasb(index, irds_dataset, num_results=NUM_RESULTS, rerank_k=RERANK_K)
bm25_rm3_tasb = build_bm25_rm3_tasb(
    index,
    irds_dataset,
    num_results=NUM_RESULTS,
    fb_terms=FB_TERMS,
    rerank_k=RERANK_K,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

## Cache core runs for DL19

In [64]:
runs_19 = {
    "BM25": run_and_cache(bm25, topics_19_run, CACHE_DIR / f"{CACHE_PREFIX}_bm25_dl19.pkl"),
    "BM25+RM3": run_and_cache(
        bm25_rm3,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_dl19.pkl",
    ),
    "BM25+TAS-B": run_and_cache(
        bm25_tasb,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_tasb_k{RERANK_K}_dl19.pkl",
    ),
    "BM25+RM3+TAS-B": run_and_cache(
        bm25_rm3_tasb,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_tasb_k{RERANK_K}_dl19.pkl",
    ),
}

evaluate_runs(runs_19, topics_19_run, qrels_19_run)

,name,map,recall_100,ndcg_cut_10
0,BM25,0.370004,0.442170,0.479540
1,BM25+RM3,0.406129,0.456336,0.524715
2,BM25+TAS-B,0.359250,0.442210,0.689030
3,BM25+RM3+TAS-B,0.367400,0.456336,0.687443


## Cache core runs for DL20

In [65]:
runs_20 = {
    "BM25": run_and_cache(bm25, topics_20_run, CACHE_DIR / f"{CACHE_PREFIX}_bm25_dl20.pkl"),
    "BM25+RM3": run_and_cache(
        bm25_rm3,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_dl20.pkl",
    ),
    "BM25+TAS-B": run_and_cache(
        bm25_tasb,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_tasb_k{RERANK_K}_dl20.pkl",
    ),
    "BM25+RM3+TAS-B": run_and_cache(
        bm25_rm3_tasb,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_tasb_k{RERANK_K}_dl20.pkl",
    ),
}

evaluate_runs(runs_20, topics_20_run, qrels_20_run)

,name,map,recall_100,ndcg_cut_10
0,BM25,0.358724,0.504658,0.493627
1,BM25+RM3,0.400468,0.548754,0.510773
2,BM25+TAS-B,0.385519,0.505561,0.657714
3,BM25+RM3+TAS-B,0.412778,0.549097,0.671603


## Cache RM3 + TAS-B sweep for DL19

In [66]:
rm3_tasb_runs_19 = {"BM25+TAS-B": runs_19["BM25+TAS-B"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3_tasb(
        index,
        irds_dataset,
        num_results=NUM_RESULTS,
        fb_terms=fb,
        rerank_k=RERANK_K,
    )
    rm3_tasb_runs_19[f"BM25+RM3(fb={fb})+TAS-B"] = run_and_cache(
        pipe,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_tasb_k{RERANK_K}_dl19.pkl",
    )

evaluate_runs(rm3_tasb_runs_19, topics_19_run, qrels_19_run)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

,name,map,recall_100,ndcg_cut_10
0,BM25+TAS-B,0.359250,0.442210,0.689030
1,BM25+RM3(fb=5)+TAS-B,0.364385,0.454594,0.686137
2,BM25+RM3(fb=10)+TAS-B,0.367400,0.456336,0.687443
3,BM25+RM3(fb=20)+TAS-B,0.372978,0.465349,0.688454
4,BM25+RM3(fb=30)+TAS-B,0.372421,0.464965,0.688710
5,BM25+RM3(fb=50)+TAS-B,0.373623,0.465664,0.688618


## Cache BM25 + RM3 recall sweep for DL19

In [67]:
bm25_rm3_runs_19 = {"BM25": runs_19["BM25"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3(index, num_results=NUM_RESULTS, fb_terms=fb)
    bm25_rm3_runs_19[f"BM25+RM3(fb={fb})"] = run_and_cache(
        pipe,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_dl19.pkl",
    )

evaluate_runs(
    bm25_rm3_runs_19,
    topics_19_run,
    qrels_19_run,
    eval_metrics=["recall_100"],
)

,name,recall_100
0,BM25,0.442170
1,BM25+RM3(fb=5),0.454525
2,BM25+RM3(fb=10),0.456336
3,BM25+RM3(fb=20),0.465349
4,BM25+RM3(fb=30),0.464965
5,BM25+RM3(fb=50),0.465664


## Cache RM3 + TAS-B sweep for DL20

In [68]:
rm3_tasb_runs_20 = {"BM25+TAS-B": runs_20["BM25+TAS-B"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3_tasb(
        index,
        irds_dataset,
        num_results=NUM_RESULTS,
        fb_terms=fb,
        rerank_k=RERANK_K,
    )
    rm3_tasb_runs_20[f"BM25+RM3(fb={fb})+TAS-B"] = run_and_cache(
        pipe,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_tasb_k{RERANK_K}_dl20.pkl",
    )

evaluate_runs(rm3_tasb_runs_20, topics_20_run, qrels_20_run)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

,name,map,recall_100,ndcg_cut_10
0,BM25+TAS-B,0.385519,0.505561,0.657714
1,BM25+RM3(fb=5)+TAS-B,0.415240,0.552288,0.674037
2,BM25+RM3(fb=10)+TAS-B,0.412778,0.549097,0.671603
3,BM25+RM3(fb=20)+TAS-B,0.414257,0.549762,0.676079
4,BM25+RM3(fb=30)+TAS-B,0.415657,0.551131,0.676006
5,BM25+RM3(fb=50)+TAS-B,0.416002,0.551416,0.675926


## Cache BM25 + RM3 recall sweep for DL20

In [69]:
bm25_rm3_runs_20 = {"BM25": runs_20["BM25"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3(index, num_results=NUM_RESULTS, fb_terms=fb)
    bm25_rm3_runs_20[f"BM25+RM3(fb={fb})"] = run_and_cache(
        pipe,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_dl20.pkl",
    )

evaluate_runs(
    bm25_rm3_runs_20,
    topics_20_run,
    qrels_20_run,
    eval_metrics=["recall_100"],
)

,name,recall_100
0,BM25,0.504658
1,BM25+RM3(fb=5),0.551599
2,BM25+RM3(fb=10),0.548754
3,BM25+RM3(fb=20),0.549578
4,BM25+RM3(fb=30),0.550948
5,BM25+RM3(fb=50),0.551233


## Next step

Open `analysis.ipynb` after these caches have been created.